In [ ]:
!nvidia-smi -l 1

In [ ]:
%pip install transformer_lens
%pip install circuitsvis

In [ ]:
import sys
sys.path.append('/home/galk/LanguageDynamics/src') 

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from tqdm import tqdm

import os
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.distributions.kl import kl_divergence
from torch.distributions.categorical import Categorical
from torch.distributions.multivariate_normal import MultivariateNormal
# from datasets import load_dataset
from datetime import datetime

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

# import interpretability stuff
import transformer_lens.utils as utils
from transformer_lens.hook_points import (
    HookPoint,
)  # Hooking utilities
from transformer_lens import HookedTransformer, FactoredMatrix
import circuitsvis as cv

from importlib import reload

models_path = '/home/galk/LanguageDynamics/models/linguistic_flip_flop'

In [ ]:
# Load model
device = utils.get_device()
model_name = "attn-only-2l"
model = HookedTransformer.from_pretrained(model_name, device=device)

In [ ]:
# Run with cache
prompt = "I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know. I don't know."
input_tokens = model.to_tokens(prompt)
# print(input_tokens.device)
logits, cache = model.run_with_cache(input_tokens, remove_batch_dim=True)

In [ ]:
# Get attention patterns and visualize them
layer = 1
print(type(cache))
attention_pattern = cache["attn", layer]
print(attention_pattern.shape) # [head_idx, destination, source]
str_tokens = model.to_str_tokens(prompt)

In [ ]:
print(f"Layer {layer} Head Attention Patterns:")
cv.attention.attention_patterns(tokens=str_tokens, attention=attention_pattern)

In [ ]:
jump = 5
offset = 606
plt.plot(attention_pattern[6, offset].cpu().numpy(), '--o')
# plt.plot(attention_pattern[6, offset].cpu().numpy()[offset-jump+1::-jump][::-1], '--o')
plt.ylim(bottom=0)
# plt.show()

# print(attention_pattern[6, -1].cpu().numpy().sum())

relative_coeffs = [attention_pattern[6, pos].cpu().numpy()[pos-jump+1::-jump].sum() / attention_pattern[6, pos].cpu().numpy().sum() for pos in np.arange(6, 641, jump)]
# print(attention_pattern[6, offset].cpu().numpy()[offset-jump+1::-jump].sum() / attention_pattern[6, offset].cpu().numpy()[0])
plt.figure()
plt.plot(relative_coeffs)
plt.xlabel("#Repeats")
plt.ylabel("Relative Coeff")
plt.grid()
plt.show()

In [ ]:
layer = 1

# 1. The Residual Stream
# shape: [batch, sequence_length, d_model]
resid_pre = cache[f"blocks.{layer}.hook_resid_pre"] # Before attention
resid_post = cache[f"blocks.{layer}.hook_resid_post"] # After attention & addition

# 2. The Q, K, and V Projections
# shape: [batch, sequence_length, num_heads, d_head]
q_vectors = cache[f"blocks.{layer}.attn.hook_q"] # shape [position, head_idx, d_head]
k_vectors = cache[f"blocks.{layer}.attn.hook_k"]
v_vectors = cache[f"blocks.{layer}.attn.hook_v"]
z_vectors = cache[f"blocks.{layer}.attn.hook_z"]

# 3. Attention Scores
# hook_attn_scores: Unnormalized dot products (Q*K^T)
# hook_pattern: Normalized probabilities (after Softmax)
# shape: [batch, num_heads, query_pos, key_pos]
unnormalized_scores = cache[f"blocks.{layer}.attn.hook_attn_scores"]
attention_pattern = cache[f"blocks.{layer}.attn.hook_pattern"]

# 4. Computed Outputs (Before being added to residual stream)
# hook_z: The output of the OV circuit before the final W_O projection
# hook_result: The final output of each head after W_O projection (the exact H_t we modeled)
# shape (hook_result): [batch, sequence_length, num_heads, d_model]
# head_outputs = cache[f"blocks.{layer}.attn.hook_result"]

# 5. Final Logits
# Already returned from run_with_cache. shape: [batch, sequence_length, vocab_size]
print(f"Logits shape: {logits.shape}")

In [ ]:
HEAD_IDX = 6
offset = 0
jump = 5
z_norms = np.linalg.norm(z_vectors[:,HEAD_IDX].cpu().numpy(), axis=-1)
z_norms = z_norms[1:] # remove the <BOS> token

plt.figure(figsize=(10, 8))
plt.plot(np.reshape(z_norms, (-1, jump)))
# plt.plot(z_norms[offset::jump])
plt.grid()
plt.xlabel("Position")
plt.ylabel("Z norm")
plt.ylim(bottom=0)
plt.show()